# Flash Flash Revolution Silver Layer Directional Complexity Features

This notebook calculates **note-level directional complexity features** that capture arrow pattern difficulty. These features address a critical gap in the original model - the complete absence of orientation/directional information.

## Features Computed

### 1. orientation_changed (Binary: 0 or 1)
**Hypothesis:** Direction switches between consecutive notes increase difficulty
- **Value 1**: Current note has different orientation than previous note
- **Value 0**: Same orientation as previous note OR first note in song
- **Why it matters**: Rapid direction changes require faster cognitive processing and physical adaptation

### 2. is_single_arrow (Binary: 0 or 1)
**Hypothesis:** Multi-arrow presses (jumps) are more difficult than single arrows
- **Value 1**: Single arrow press (e.g., '0001', '0010', '0100', '1000')
- **Value 0**: Multiple arrows pressed simultaneously (e.g., '0011', '1100', '1010')
- **Detection method**: Count number of '1's in orientation string
- **Why it matters**: Jumps require coordination and are physically more demanding

### 3. double_jump (Binary: 0 or 1)
**Hypothesis:** Two-arrow jumps are the most common multi-arrow pattern
- **Value 1**: Exactly 2 arrows pressed (e.g., '0011', '1010')
- **Value 0**: Single arrow or 3+ arrows
- **Why it matters**: Double jumps are distinct difficulty tier between singles and triple/quad jumps

### 4. triple_quad_jump (Binary: 0 or 1)
**Hypothesis:** Three or four arrow presses are extremely difficult
- **Value 1**: 3 or 4 arrows pressed simultaneously
- **Value 0**: Fewer than 3 arrows
- **Why it matters**: Rare patterns that significantly spike difficulty

---

## Incremental Processing

**Change Detection:**
- Monitors `swf_version` from bronze__songlist to detect song updates
- Only processes songs that are new or have changed
- Uses DELETE + INSERT pattern for incremental updates

**Dependencies:**
- `acubed.ffr.silver__zero-framer` (decomposed notes with orientations)
- `acubed.ffr.bronze__songlist` (swf_version for change detection)

---

## Output Table

**Table:** `acubed.ffr.silver__directional-complexity`

**Schema:**
- `song_id` (bigint): Song identifier
- `note_id` (int): Note sequence number
- `orientation` (string): Arrow pattern (e.g., '0001', '1100')
- `orientation_changed` (int): 1 if direction changed from previous note
- `is_single_arrow` (int): 1 if single arrow press
- `double_jump` (int): 1 if exactly 2 arrows
- `triple_quad_jump` (int): 1 if 3 or 4 arrows
- `swf_version` (bigint): Version tracking

**Granularity:** One row per note

In [0]:
from pyspark.sql.functions import (
    col, lag, when, length, regexp_replace, lit
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable

print("✓ Imports loaded successfully")

In [0]:
# Configuration: Automatic Processing Mode
# Automatically determines whether to do full refresh or incremental processing
# - Full refresh (process_all=True): When table doesn't exist (first run)
# - Incremental (process_all=False): When table exists (subsequent runs)

silver_table = "acubed.ffr.`silver__directional-complexity`"
process_all = not spark.catalog.tableExists(silver_table)

if process_all:
    print(f"ℹ Auto-detected: Full refresh mode (table does not exist)")
else:
    print(f"ℹ Auto-detected: Incremental mode (table exists)")

print(f"✓ Configuration loaded: process_all = {process_all}")

In [0]:
# Identify songs that need to be processed (new or updated)
# Uses swf_version from bronze__songlist to detect changes

silver_table = "acubed.ffr.`silver__directional-complexity`"

if spark.catalog.tableExists(silver_table) and not process_all:
    # Silver table exists - check for new songs and updated songs
    bronze_songlist = spark.table("acubed.ffr.bronze__songlist").select(
        col("id").alias("song_id"),
        col("swf_version").alias("bronze_swf_version")
    )
    
    silver_directional = spark.table(silver_table).select(
        col("song_id").alias("silver_song_id"),
        col("swf_version").alias("silver_swf_version")
    ).distinct()
    
    # Left join to find new songs and updated songs
    changed_songs = bronze_songlist.join(
        silver_directional,
        bronze_songlist.song_id == silver_directional.silver_song_id,
        "left"
    ).filter(
        # New songs (not in silver) OR updated songs (different swf_version)
        col("silver_song_id").isNull() |
        (col("bronze_swf_version") != col("silver_swf_version"))
    )
    
    changed_song_ids = changed_songs.select(col("song_id")).distinct()
    num_changed = changed_song_ids.count()
    
    if num_changed == 0:
        print("ℹ No changed songs detected")
        changed_song_ids = None
    else:
        print(f"✓ Found {num_changed} songs to process (new or updated)")
else:
    # Force full refresh or table doesn't exist
    print("✓ Will process all songs")
    changed_song_ids = None

In [0]:
# Guard: Skip if no songs to process
if not process_all and changed_song_ids is None:
    print("ℹ No songs to process - skipping directional complexity calculation")
    print("✓ Notebook will complete successfully (idempotent run)")
    # Set empty DataFrame
    df_directional = spark.createDataFrame([], schema="""
        song_id bigint, note_id int, orientation string,
        orientation_changed int, is_single_arrow int, double_jump int,
        triple_quad_jump int, swf_version bigint
    """)
else:
    print("="*60)
    print("COMPUTING DIRECTIONAL COMPLEXITY FEATURES")
    print("="*60)
    
    # Load decomposed notes from silver__notes-adjusted and join with swf_version
    df_notes = spark.table("acubed.ffr.`silver__notes-adjusted`").alias("n").join(
        spark.table("acubed.ffr.bronze__songlist").select(
            col("id").alias("songlist_id"),
            col("swf_version")
        ).alias("s"),
        col("n.song_id") == col("s.songlist_id"),
        "inner"
    ).select(
        col("n.song_id"),
        col("n.note_id"),
        col("n.orientation"),
        col("s.swf_version")
    )
    
    # Filter to changed songs if doing incremental processing
    if not process_all and changed_song_ids is not None:
        df_notes = df_notes.join(changed_song_ids, "song_id", "inner")
        print(f"ℹ Incremental mode: Processing {df_notes.select('song_id').distinct().count()} changed songs")
    else:
        print(f"ℹ Full refresh mode: Processing all songs")
    
    # Define window for lag operation (previous note in same song)
    window_spec = Window.partitionBy("song_id").orderBy("note_id")
    
    # Calculate features
    df_directional = df_notes.withColumn(
        "prev_orientation",
        lag(col("orientation")).over(window_spec)
    ).withColumn(
        # Feature 1: Orientation changed from previous note
        "orientation_changed",
        when(
            (col("prev_orientation").isNotNull()) & 
            (col("orientation") != col("prev_orientation")),
            1
        ).otherwise(0)
    ).withColumn(
        # Count number of '1's in orientation to determine arrow count
        "arrow_count",
        length(regexp_replace(col("orientation"), "0", ""))
    ).withColumn(
        # Feature 2: Single arrow press
        "is_single_arrow",
        when(col("arrow_count") == 1, 1).otherwise(0)
    ).withColumn(
        # Feature 3: Double jump (exactly 2 arrows)
        "double_jump",
        when(col("arrow_count") == 2, 1).otherwise(0)
    ).withColumn(
        # Feature 4: Triple or quad jump (3 or 4 arrows)
        "triple_quad_jump",
        when(col("arrow_count") >= 3, 1).otherwise(0)
    ).select(
        "song_id", "note_id", "orientation",
        "orientation_changed", "is_single_arrow",
        "double_jump", "triple_quad_jump",
        "swf_version"
    )
    
    row_count = df_directional.count()
    print(f"\n✓ Calculated directional complexity for {row_count:,} notes")
    print("\nFeatures computed:")
    print("  1. orientation_changed - direction switches")
    print("  2. is_single_arrow - single vs multi-arrow")
    print("  3. double_jump - two-arrow jumps")
    print("  4. triple_quad_jump - three/four-arrow jumps")

In [0]:
# Save to Delta table using DELETE + INSERT pattern for incremental updates
table_name = "acubed.ffr.`silver__directional-complexity`"

# Capture counts BEFORE operations
rows_to_insert = df_directional.count()

if rows_to_insert == 0:
    print("ℹ No rows to insert - skipping table update")
    print("✓ Table remains unchanged (idempotent run)")
else:
    print("\n" + "="*60)
    print("SAVING TO DELTA TABLE")
    print("="*60)
    
    if spark.catalog.tableExists(table_name):
        # Table exists - use DELETE + INSERT for incremental updates
        delta_table = DeltaTable.forName(spark, table_name)
        
        # Get list of song_ids being updated
        updated_song_ids = df_directional.select("song_id").distinct()
        song_ids_to_delete = [row.song_id for row in updated_song_ids.collect()]
        
        if len(song_ids_to_delete) > 0:
            # Build condition for DELETE
            delete_condition = col("song_id").isin(song_ids_to_delete)
            delta_table.delete(delete_condition)
            print(f"✓ Deleted existing rows for {len(song_ids_to_delete)} songs")
        
        # Insert new rows
        df_directional.write.format("delta").mode("append").saveAsTable(table_name)
        print(f"✓ Inserted {rows_to_insert:,} new rows")
        
        # Get final count
        final_count = spark.table(table_name).count()
        print(f"\n✓ Updated {table_name}")
        print(f"  Total rows: {final_count:,}")
    else:
        # Table doesn't exist - create it
        df_directional.write.format("delta").mode("overwrite").saveAsTable(table_name)
        print(f"✓ Created {table_name} with {rows_to_insert:,} rows")
    
    print("\n" + "="*60)
    print("✓ Directional complexity features ready!")
    print("="*60)

In [0]:
%sql
SELECT 
  song_id,
  COUNT(*) as total_notes,
  SUM(orientation_changed) as direction_changes,
  SUM(is_single_arrow) as single_arrows,
  SUM(double_jump) as double_jumps,
  SUM(triple_quad_jump) as triple_quad_jumps,
  ROUND(SUM(orientation_changed) * 100.0 / COUNT(*), 1) as pct_direction_changes
FROM acubed.ffr.`silver__directional-complexity`
GROUP BY song_id
ORDER BY total_notes DESC
LIMIT 20